# ADL 원시 데이터 조회

`adl_raw_records` 테이블의 데이터를 간단히 조회하는 노트북. Tortoise ORM(async)을 사용한다.

데이터가 없으면 먼저 `adl_raw_ingest.ipynb` 로 적재한다.

## 1. 환경 설정 + DB 연결

In [1]:
import os
import sys
from pathlib import Path

# 작업 디렉토리를 프로젝트 루트로 이동 (.env.local·app import 해석용).
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from tortoise import Tortoise

from app.config import settings
from app.database import TORTOISE_ORM
from app.models.adl_raw import AdlRawRecord

pd.set_option("display.max_columns", 60)

if not Tortoise.apps:
    await Tortoise.init(config=TORTOISE_ORM)

print("DB 연결:", settings.DATABASE_URL.split("@")[-1])

DB 연결: db.salpyeobom.kro.kr:15432/spb_db


## 2. 전체 건수 + source_type 분포

In [2]:
total = await AdlRawRecord.all().count()
emergency = await AdlRawRecord.filter(source_type="응급").count()
death = await AdlRawRecord.filter(source_type="사망").count()

print(f"전체: {total}건  (응급: {emergency}, 사망: {death})")

전체: 60건  (응급: 30, 사망: 30)


## 3. 상위 10건 미리보기

1440-길이 배열 컬럼은 너무 길어 표 가독성을 해치므로 제외하고 본다.

In [3]:
EXCLUDE_ARRAY = [
    "place_code_1_list", "aix_1_list", "aix_h_list",
    "sleep_depth_1_list", "outgoing_1_list",
    "temp_list", "humi_list", "illu_list",
]

rows = await AdlRawRecord.all().order_by("id").limit(10).values()
df = pd.DataFrame(rows).drop(columns=EXCLUDE_ARRAY, errors="ignore")
df

,id,source_type,care_recipient_id,age,sex,alone,vision,hearing,dosage,district,house_structure,room_no,bath_location,lifeog_date,emergency_date,emergency_record,occurrence_place,on_site,hospital_transfer,hospital_treatment,death_date,death_record,aix_d,aix_1_eq_0_repeat_count,total_aix_sum,total_aix_inc_ratio,night_aix_ratio,total_age_aix_ratio,sleep_start_time_d,sleep_end_time_d,total_sleep_period,total_sleep_aix_ratio,bath_count_d,bath_time_d,bath_nomove_time,bath_count_in_sleep,bath_time_per_count,total_bath_average_count,outgoing_count_d,outgoing_time_d,outgoing_late_night_count_d,outgoing_late_night_time_d,last_outgoing_time,total_outgoing_average_time,total_outgoing_average_count,created_at
0,1,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2022-01-03,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,36.0,0,75.0,-52.0,1828.0,36.0,45,694,63.0,70.0,8,161.0,None,3,20.0,15.0,9,444.0,0,0.0,0,375.0,6.0,2026-05-25 12:06:40.887352+00:00
1,2,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2022-01-02,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,25.0,0,73.0,-65.0,0.0,25.0,38,674,436.0,65.0,11,142.0,None,2,12.0,15.0,11,793.0,1,240.0,0,383.0,7.0,2026-05-25 12:06:40.887352+00:00
2,3,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2022-01-01,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,25.0,0,72.0,-65.0,3797.0,25.0,33,639,130.0,65.0,10,194.0,None,3,19.0,15.0,8,255.0,0,0.0,0,374.0,7.0,2026-05-25 12:06:40.887352+00:00
3,4,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-31,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,59.0,0,71.0,-16.0,7959.0,59.0,36,572,100.0,81.0,7,220.0,None,2,31.0,14.0,3,349.0,0,0.0,0,379.0,7.0,2026-05-25 12:06:40.887352+00:00
4,5,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-30,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,33.0,0,70.0,-52.0,4262.0,33.0,170,549,719.0,75.0,5,14.0,None,2,2.0,14.0,10,367.0,0,0.0,0,389.0,7.0,2026-05-25 12:06:40.887352+00:00
5,6,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-29,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,55.0,0,69.0,-20.0,20417.0,55.0,13,365,139.0,66.0,6,27.0,None,0,4.0,14.0,5,813.0,0,0.0,0,408.0,7.0,2026-05-25 12:06:40.887352+00:00
6,7,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-28,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,75.0,0,69.0,8.0,13113.0,75.0,40,620,100.0,81.0,5,109.0,None,1,21.0,14.0,8,340.0,0,0.0,0,406.0,7.0,2026-05-25 12:06:40.887352+00:00
7,8,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-27,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,33.0,0,69.0,-52.0,764.0,33.0,356,474,0.0,0.0,6,99.0,None,1,16.0,14.0,12,721.0,1,167.0,0,422.0,8.0,2026-05-25 12:06:40.887352+00:00
8,9,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-26,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,31.0,0,68.0,-54.0,3596.0,31.0,1,574,718.0,68.0,11,57.0,None,3,5.0,13.0,13,287.0,1,32.0,0,426.0,8.0,2026-05-25 12:06:40.887352+00:00
9,10,응급,661,78,F,Y,보통,보통,없음,도시,주택,2,옥내,2021-12-25,2022-01-03,09:10 휴대폰 / 부재2회\n09:10 G/W / 요양보호사 / 계단에서 넘어져...,실외,출동,이송,입원,None,None,33.0,0,67.0,-50.0,3993.0,33.0,41,669,0.0,0.0,8,145.0,None,4,18.0,13.0,12,603.0,0,0.0,0,423.0,8.0,2026-05-25 12:06:40.887352+00:00


## 4. care_recipient_id 로 필터 조회

특정 대상자의 모든 행을 `lifeog_date` 순으로 본다.

In [4]:
TARGET_ID = "661"  # 원하는 care_recipient_id 로 변경

rows = (
    await AdlRawRecord.filter(care_recipient_id=TARGET_ID)
    .order_by("lifeog_date")
    .values(
        "id", "source_type", "care_recipient_id", "lifeog_date",
        "emergency_date", "death_date",
        "aix_d", "total_aix_sum",
        "total_sleep_period", "bath_count_d", "outgoing_count_d",
    )
)
pd.DataFrame(rows)

,id,source_type,care_recipient_id,lifeog_date,emergency_date,death_date,aix_d,total_aix_sum,total_sleep_period,bath_count_d,outgoing_count_d
0,30,응급,661,2021-12-05,2022-01-03,None,252.0,66.0,0.0,21,7
1,29,응급,661,2021-12-06,2022-01-03,None,251.0,63.0,0.0,48,6
2,28,응급,661,2021-12-07,2022-01-03,None,323.0,58.0,0.0,31,7
3,27,응급,661,2021-12-08,2022-01-03,None,107.0,57.0,0.0,30,16
4,26,응급,661,2021-12-09,2022-01-03,None,127.0,56.0,69.0,16,6
5,25,응급,661,2021-12-10,2022-01-03,None,57.0,56.0,0.0,9,9
6,24,응급,661,2021-12-11,2022-01-03,None,51.0,56.0,38.0,8,11
7,23,응급,661,2021-12-12,2022-01-03,None,30.0,57.0,506.0,9,9
8,22,응급,661,2021-12-13,2022-01-03,None,49.0,57.0,603.0,12,11
9,21,응급,661,2021-12-14,2022-01-03,None,89.0,56.0,44.0,35,12


## 5. 배열 컬럼 한 건 확인

8개 배열 컬럼은 PostgreSQL 네이티브 배열이라 `.values()` 결과에 Python `list` 로 들어온다.
길이와 앞 8개 원소만 본다.

In [5]:
record = await AdlRawRecord.first()
print(f"id={record.id}, source_type={record.source_type}, "
      f"care_recipient_id={record.care_recipient_id}, lifeog_date={record.lifeog_date}")
print()

for col in EXCLUDE_ARRAY:
    v = getattr(record, col)
    length = len(v) if isinstance(v, list) else None
    head = v[:8] if isinstance(v, list) else v
    print(f"{col}: 길이={length}, 앞부분={head}")

id=1, source_type=응급, care_recipient_id=661, lifeog_date=2022-01-03

place_code_1_list: 길이=1440, 앞부분=[0, 0, 0, 0, 0, 0, 0, 0]
aix_1_list: 길이=1440, 앞부분=[0, 0, 0, 0, 0, 0, 0, 0]
aix_h_list: 길이=24, 앞부분=[18, 7, 4, 0, 52, 311, 110, 13]
sleep_depth_1_list: 길이=1440, 앞부분=[4, 4, 4, 4, 4, 4, 4, 4]
outgoing_1_list: 길이=1440, 앞부분=[255, 254, 254, 254, 254, 254, 254, 254]
temp_list: 길이=24, 앞부분=[30.0, 30.0, 29.8, 29.6, 29.4, 29.2, 29.1, 29.0]
humi_list: 길이=24, 앞부분=[59.7, 59.6, 60.1, 60.4, 60.8, 61.2, 61.4, 61.8]
illu_list: 길이=24, 앞부분=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 10.0]


## 6. 연결 종료

In [6]:
await Tortoise.close_connections()
print("DB 연결 종료")

DB 연결 종료
